# Train the Aircraft Vision model (free Colab GPU)

1. **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.
2. Training EfficientNet-B0 on FGVC-Aircraft (100 types) takes roughly 40–60 min on a T4.
3. The last cell downloads `aircraft_model.zip`. Unzip it into `android/app/src/main/assets/`.

Mounting Google Drive (optional cell below) keeps the dataset and checkpoints if Colab disconnects. Rerun the training cell with `--resume` to continue.

In [ ]:
!git clone https://github.com/Themagicianleo/aircraft-vision.git
%cd aircraft-vision/training
!pip -q install timm onnx onnxruntime onnxscript
!nvidia-smi --query-gpu=name,memory.total --format=csv

Optional: keep data and checkpoints on Google Drive so a disconnect doesn't lose progress.

In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/aircraft-vision'
else:
    WORK = '/content/work'
import os; os.makedirs(WORK, exist_ok=True); print(WORK)

## Train
Downloads FGVC-Aircraft (~2.7 GB) on first run. Add `--extra-data /content/my_photos` to include your own photos
(layout: `my_photos/train/<type>/*.jpg`, `my_photos/val/<type>/*.jpg`). That's how to add types FGVC lacks, like the 787 and A350.

In [ ]:
!python train.py --data-root {WORK}/data --out {WORK}/runs/effb0 --workers 2 --epochs 40 --img-size 320

## Export to ONNX (fp32 + int8) and measure accuracy with the app's exact preprocessing

In [ ]:
!python export.py --checkpoint {WORK}/runs/effb0/last.pt --out {WORK}/exported --int8 --calib-dir {WORK}/data/fgvc_prepared/images
!python evaluate.py --onnx {WORK}/exported/aircraft.onnx --data-root {WORK}/data
!python evaluate.py --onnx {WORK}/exported/aircraft_int8.onnx --labels {WORK}/exported/labels.json --data-root {WORK}/data

## Download
Pick the int8 model if its accuracy above is within ~1% of fp32 (it's 4× smaller). Otherwise keep fp32.

In [ ]:
USE_INT8 = False
import shutil, os
pkg = '/content/aircraft_model'
os.makedirs(pkg, exist_ok=True)
shutil.copy(f'{WORK}/exported/' + ('aircraft_int8.onnx' if USE_INT8 else 'aircraft.onnx'), f'{pkg}/aircraft.onnx')
shutil.copy(f'{WORK}/exported/labels.json', f'{pkg}/labels.json')
shutil.make_archive('/content/aircraft_model', 'zip', pkg)
from google.colab import files
files.download('/content/aircraft_model.zip')